# check the spectrum (pfscoadd) from a given release
This notebook:
1. reads `object_id` and `gr_id` from an ECSV file,
2. gets `pfsObject` and best-fit 1drp `model` for each `(object_id, gr_id)`
- using butler directly
- read pfscoadd fits file directly

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.table import join
from astropy.table import vstack
from pathlib import Path
from astropy.table import Table

from lsst.daf.butler import Butler
from pfs.datamodel import TargetType
from pfs.datamodel import FiberStatus
from pfs.datamodel.pfsTargetSpectra import PfsTargetSpectra
from pfs.datamodel import PfsCoZCandidates, PfsZCandidates

import yaml
from pfs.datamodel.masks import MaskHelper
from scipy import ndimage

import matplotlib.pyplot as plt
%matplotlib inline

# basic set up for one release

In [ ]:
catId = 10091 # catId for SSP Cosmology scientific targets (note: this includes ancillary targets as well)

# SP
#pfs_base_dir = "/shared/pfs/programs/"
# idark
pfs_base_dir = "/lustre/work/jingjing.shi/PFS_SSP/hscpfs.mtk.nao.ac.jp/fileaccess/pfs/programs/"
semester_code = 'S25B-OT02'
collections = ['run25_February2026'] # release collection
release = collections[0]

butler = Butler(pfs_base_dir + semester_code + '/2d/', collections=collections)
combination = 'selected_run25' 

# get the object group map: objId and objGroup map 
# objId that belong to the same objGroup are saved in on pfsCoadd file; note this map is changing for different release
ogm = butler.get("objectGroupMap", combination=combination, cat_id=catId)

objid_release = ogm.objId
groupid_release = ogm.objGroup
nobj_release = len(objid_release)

print(f"There are {nobj_release} objects in run25, {len(np.unique(groupid_release))} groups.")

# get the unique groupid_release and rank them by ascending order of the value
unique_groupids = np.unique(groupid_release)
unique_groupids.sort()
print(f"Unique groupids: {unique_groupids}.")

In [ ]:
# for whatever object_id you want to check
fn_tmp = f"ssp_co_targets_{release}_lam1d_obs_combined.ecsv" 
table_tmp = Table.read(fn_tmp)
object_id_tmp = table_tmp['object_id'][:5]

# one-to-one mapping from objid_release -> groupid_release
obj_to_group = dict(zip(objid_release, groupid_release))
missing_obj = [oid for oid in object_id_tmp if oid not in obj_to_group]
if missing_obj:
    raise ValueError(f"object_id not found in release map: {missing_obj[:5]}")

group_id_tmp = np.array([obj_to_group[oid] for oid in object_id_tmp])

# get the spectrum (pfscoadds) of the object_id list
Two methods:
1. get the spectrum using butler directly (slow), good if the object_id list is not long
2. read the spectrum from pfscoadds fits file, fast but needs to follow the datamodel updates

## using butler

In [ ]:
def get_pfsobject_and_model(
    butler,
    combination,
    catId,
    object_id,
    gr_id,
    lam1d_base
):
    """Return (pfsObject, zCand) for one (object_id, gr_id)."""
    pfsCoadd = butler.get(
        "pfsCoadd",
        combination=combination,
        cat_id=int(catId),
        obj_group=int(gr_id),
    )

    pfsObject = pfsCoadd[int(catId), int(object_id)]
    group_dir = f"{int(catId)}_{int(gr_id)}"

    lam1d_path = (
        Path(lam1d_base)
        / group_dir
        / "data"
        / f"pfsCoZcandidates-{int(catId)}.fits"
    )

    zCand = PfsCoZCandidates.readFits(str(lam1d_path))[int(catId), int(object_id)]

    return pfsObject, zCand

def _good_mask(pfsObject):
    return ~(pfsObject.mask & pfsObject.flags.get("BAD", "CR", "SAT", "NO_DATA") != 0)

def plot_spectrum_w_1d_model(pfsObject, zCand):
    # good pixel mask
    good = _good_mask(pfsObject)

    # 1drp best model and redshift
    model = zCand.get_classified_model()
    z_lam1d = zCand.galaxy.parameters[0]['redshift']
    
    plt.plot(pfsObject.wavelength[good], pfsObject.flux[good], label="observed spectrum", color='gray', alpha=0.4, linewidth=0.2)
    plt.plot(pfsObject.wavelength[good], ndimage.median_filter(pfsObject.flux[good], size=10), '-', alpha=0.6, linewidth=0.3, color='black', label='binned spectrum')
    plt.plot(pfsObject.wavelength, model, label="Best model", color='red', alpha=1, linewidth=1)
    plt.plot(pfsObject.wavelength[good], np.sqrt(pfsObject.variance[good]), '--', label="noise", color='yellow', alpha=0.4, linewidth=0.2)
    
    frac = np.nanpercentile(pfsObject.flux, (5., 95.))
    plt.ylim(frac[0]*0.9, frac[1]*1.5)
    #plt.ylim(-5000, 10000)
        
    OII = np.array([3727.092, 3729.875]) / 10.0
    if z_lam1d != None:
        plt.axvline(x=(1+z_lam1d)*OII[0])
        plt.axvline(x=(1+z_lam1d)*OII[1])
    
    # best model recognized lines
    y_pos = frac[1]*0.8
    for x, xname in zip(zCand.galaxy.lines['lineWave'], zCand.galaxy.lines['lineName']):
        plt.text(x, y_pos, f"{xname}", rotation=90, ha='center', va='center', color='red', alpha=0.5)

    #print(np.unique(pfsObject.observations.visit))
    
    plt.xlabel('wavelength [nm]')
    plt.ylabel('flux [nJy]')
    plt.legend()
    plt.title(f'objId={zCand.target.objId}, z_lam1d={z_lam1d:.5f}')
    plt.show()
    plt.close()

In [ ]:
# example usage
pfsObject, zCand = get_pfsobject_and_model(
    butler=butler,
    combination=combination,
    catId=catId,
    object_id=object_id_tmp[0],
    gr_id=group_id_tmp[0],
    lam1d_base=f"{pfs_base_dir}/{semester_code}/lam1d/{release}/modified/"
)
plot_spectrum_w_1d_model(pfsObject, zCand)

## reading pfscoadds fits directly

In [ ]:
'''
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     475   ()      
  1  TARGET        1 BinTableHDU     25   1756R x 8C   ['I', 'J', 'J', 'PA(3)', 'K', 'D', 'D', 'I']   
  2  TARGETFLUX    1 BinTableHDU     15   8772R x 3C   [I, PA(7), E]   
  3  OBSERVATIONS    1 BinTableHDU     29   21072R x 10C   [I, J, PA(1), I, K, J, 2D, 2D, PA(29), E]   
  4  WAVELENGTH    1 ImageHDU         7   (11501,)   float32   
  5  FLUX          1 ImageHDU         8   (11501, 1756)   float32   
  6  MASK          1 CompImageHDU      8   (11501, 1756)   int32   
  7  SKY           1 ImageHDU         8   (11501, 1756)   float32   
  8  COVAR         1 BinTableHDU     15   1756R x 3C   [PE(11501), PE(11501), PE(11501)]   
  9  COVAR2        1 ImageHDU         9   (1, 1, 1756)   float64   
 10  METADATA      1 BinTableHDU     13   1756R x 2C   [I, PA(518)]   
 11  FLUXTABLE     1 BinTableHDU     19   1756R x 5C   [I, QD(12448), QE(12448), QE(12448), QJ(12448)]   
 12  NOTES         1 BinTableHDU     11   0R x 0C   []   
'''

def read_pfscoadds_fits_file(filename):
    hdul = fits.open(filename)
    objId = hdul['TARGET'].data['objId']
    #targetId = hdul['TARGET'].data['targetId']
    #ra = hdul['TARGET'].data['ra']
    #dec = hdul['TARGET'].data['dec']
    wavelength = hdul['WAVELENGTH'].data
    flux = hdul['FLUX'].data
    mask = hdul['MASK'].data
    #sky = hdul['SKY'].data
    metadata = hdul['METADATA'].data
    variance = hdul['COVAR'].data['row_0']

    pfscoadd = dict()
    pfscoadd['objid'] = objId
    pfscoadd['wavelength'] = wavelength
    pfscoadd['flux'] = flux
    pfscoadd['mask'] = mask
    pfscoadd['metadata'] = metadata
    pfscoadd['variance'] = variance
    
    return pfscoadd


def get_flags(metadataRow):
    metadata_tmp = yaml.load(
                    # This complicated conversion is required in order to preserve the newlines
                    "".join(np.char.decode(metadataRow['metadata'].astype("S"))),
                    Loader=yaml.SafeLoader,
                )
    flags = MaskHelper.fromFitsHeader(metadata_tmp, strip=True)
    return flags


def plot_spectrum(objid_sel, wavelength, flux, variance, good, groupid):
    plt.plot(wavelength[good], flux[good], label="observed spectrum", color='gray', alpha=0.4, linewidth=0.2)
    plt.plot(wavelength[good], np.sqrt(variance[good]), '--', label="noise", color='yellow', alpha=0.4, linewidth=0.2)
    plt.plot(wavelength[good], ndimage.median_filter(np.float32(flux[good]), size=20), '-', alpha=1.0, linewidth=0.3, color='black', label='binned spectrum')

    plt.ylim(-10000, 20000)
            
    plt.xlabel('wavelength [nm]')
    plt.ylabel('flux [nJy]')
    plt.legend()
    plt.title(f'objId: {objid_sel}, objGroup: {groupid}')
    plt.show()
    plt.close()

In [ ]:
# example usage of reading pfsCoadd fits file directly
unique_groupid_sel = np.sort(np.unique(group_id_tmp))
print(f"Number of unique group ids: {len(unique_groupid_sel)}. Unique group ids: {unique_groupid_sel}.")

for groupid in unique_groupid_sel:
    objid_this_group = object_id_tmp[group_id_tmp == groupid]
    print(f"Processing group id: {groupid} with {len(objid_this_group)} objects")

    # get the pfsCoadd file for this groupid in S25A_April2026
    pfscoadd_fn = f"{pfs_base_dir}/{semester_code}/2d/{release}/pfsCoadd/{catId}/pfsCoadd_PFS_{combination}_{catId}_{groupid}_{release}.fits"
    pfscoadds_gr = read_pfscoadds_fits_file(pfscoadd_fn)

    objId_gr, wavelength_gr, flux_gr, mask_gr, metadata_gr, variance_gr = pfscoadds_gr['objid'], pfscoadds_gr['wavelength'], pfscoadds_gr['flux'], \
                                                        pfscoadds_gr['mask'], pfscoadds_gr['metadata'], pfscoadds_gr['variance']  
    
   
    # plot the spectra 
    for objid_sel in objid_this_group:
        index = np.where(objId_gr == objid_sel)[0][0]
        metadataRow = metadata_gr[index]
        flags = get_flags(metadataRow)
        good = ~(mask_gr[index] & flags.get('BAD', 'CR', 'SAT', 'NO_DATA') != 0)

        plot_spectrum(objid_sel, wavelength_gr, flux_gr[index], variance_gr[index], good, groupid)